In [288]:
# Importação das bibliotecas
import matplotlib.pyplot as plt
import pandas as pd
from pandas import DataFrame
from pandas.io.parsers import TextFileReader

# Configuração padrão dos gráficos
plt.rcParams["figure.figsize"] = (8, 5)

# Download latest version
dados: TextFileReader | DataFrame = pd.read_csv("../data/raw/Telco_Customer_Churn.csv")

dados.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [289]:
dados.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [290]:
#Check os registros para as colunas em questão.
dados[[ "Contract"
,"tenure"
,"InternetService"
,"TechSupport"
,"OnlineSecurity"
,"MonthlyCharges"
,"PaymentMethod"
,"Dependents"]]



,Contract,tenure,InternetService,TechSupport,OnlineSecurity,MonthlyCharges,PaymentMethod,Dependents
0,Month-to-month,1,DSL,No,No,29.85,Electronic check,No
1,One year,34,DSL,No,Yes,56.95,Mailed check,No
2,Month-to-month,2,DSL,No,Yes,53.85,Mailed check,No
3,One year,45,DSL,Yes,Yes,42.30,Bank transfer (automatic),No
4,Month-to-month,2,Fiber optic,No,No,70.70,Electronic check,No
...,...,...,...,...,...,...,...,...
7038,One year,24,DSL,Yes,Yes,84.80,Mailed check,Yes
7039,One year,72,Fiber optic,No,No,103.20,Credit card (automatic),Yes
7040,Month-to-month,11,DSL,No,Yes,29.60,Electronic check,Yes
7041,Month-to-month,4,Fiber optic,No,No,74.40,Mailed check,No


In [291]:
#transformando as colunas para o formato de hot encode, ou seja, transformando as categorias em colunas binárias (0 ou 1).
colunas_para_hot_encode = [
    "Contract",
    "InternetService",
    "TechSupport",
    "OnlineSecurity",
    "PaymentMethod",
    "MultipleLines",
    "OnlineBackup",
    "DeviceProtection",
    "StreamingTV",
    "StreamingMovies",
]

dados_encoded = pd.get_dummies(
    dados,
    columns=colunas_para_hot_encode,
    dtype=int,
    drop_first=True
)

for coluna in colunas_para_hot_encode:
    if coluna in dados_encoded.columns:
        dados_encoded.drop(columns=[coluna], inplace=True)

dados_encoded.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,...,MultipleLines_No phone service,MultipleLines_Yes,OnlineBackup_No internet service,OnlineBackup_Yes,DeviceProtection_No internet service,DeviceProtection_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes
0,7590-VHVEG,Female,0,Yes,No,1,No,Yes,29.85,29.85,...,1,0,0,1,0,0,0,0,0,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,56.95,1889.5,...,0,0,0,0,0,1,0,0,0,0
2,3668-QPYBK,Male,0,No,No,2,Yes,Yes,53.85,108.15,...,0,0,0,1,0,0,0,0,0,0
3,7795-CFOCW,Male,0,No,No,45,No,No,42.30,1840.75,...,1,0,0,0,0,1,0,0,0,0
4,9237-HQITU,Female,0,No,No,2,Yes,Yes,70.70,151.65,...,0,0,0,0,0,0,0,0,0,0


In [292]:
# dados_encoded["TotalCharges"].value_counts()
#Validando que podemos assumir que o total de charges é igual o valor mensal vezes o tempo de contrato.
dados_encoded[["Churn","tenure"]][dados_encoded["TotalCharges"] == " "].value_counts()





Churn  tenure
No     0         11
Name: count, dtype: int64

In [293]:
import numpy as np

# substituir espaços por NaN
dados_encoded["TotalCharges"] = dados_encoded["TotalCharges"].replace(" ", np.nan)

# converter para número
dados_encoded["TotalCharges"] = pd.to_numeric(dados_encoded["TotalCharges"])

# encontrar valores faltantes
mask = dados_encoded["TotalCharges"].isna()

# corrigir tenure = 0
tenure_ajustado = dados_encoded["tenure"].replace(0, 1)

# calcular valor correto
dados_encoded.loc[mask, "TotalCharges"] = (
    dados_encoded.loc[mask, "MonthlyCharges"] *
    tenure_ajustado.loc[mask]
)

In [294]:
dados_encoded["TotalCharges"].value_counts()

TotalCharges
20.20      11
19.75       9
19.90       8
20.05       8
19.65       8
           ..
1990.50     1
7362.90     1
346.45      1
306.60      1
6844.50     1
Name: count, Length: 6534, dtype: int64

In [295]:
#Transformando as colunas numéricas para o intervalo de 0 a 1 utilizando o MinMaxScaler do sklearn. Ou seja, assumindo um valor relatorio para cada registro.
#
# Exemplo:
# 20 ----- 50 ----- 100
# 0 ----- 0.375 ----- 1
from sklearn.preprocessing import MinMaxScaler

# Corrigir coluna numérica
if "TotalCharges" in dados_encoded.columns:
    dados_encoded["TotalCharges"] = pd.to_numeric(
        dados_encoded["TotalCharges"],
        errors="raise"
    )

# Remover valores inválidos
dados_encoded = dados_encoded.dropna(subset=["TotalCharges"])

colunas_numericas = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
]
#Inicializando objeto MinMaxScaler
scaler = MinMaxScaler()

# Aplicando o scaler nas colunas numéricas
dados_encoded[colunas_numericas] = scaler.fit_transform(
    dados_encoded[colunas_numericas]
)


In [296]:
#Exibindo as primeiras linhas do dataset após a transformação
dados_encoded.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,...,MultipleLines_No phone service,MultipleLines_Yes,OnlineBackup_No internet service,OnlineBackup_Yes,DeviceProtection_No internet service,DeviceProtection_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes
0,7590-VHVEG,Female,0,Yes,No,0.013889,No,Yes,0.115423,0.001275,...,1,0,0,1,0,0,0,0,0,0
1,5575-GNVDE,Male,0,No,No,0.472222,Yes,No,0.385075,0.215867,...,0,0,0,0,0,1,0,0,0,0
2,3668-QPYBK,Male,0,No,No,0.027778,Yes,Yes,0.354229,0.010310,...,0,0,0,1,0,0,0,0,0,0
3,7795-CFOCW,Male,0,No,No,0.625000,No,No,0.239303,0.210241,...,1,0,0,0,0,1,0,0,0,0
4,9237-HQITU,Female,0,No,No,0.027778,Yes,Yes,0.521891,0.015330,...,0,0,0,0,0,0,0,0,0,0


In [297]:
dados_encoded

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,...,MultipleLines_No phone service,MultipleLines_Yes,OnlineBackup_No internet service,OnlineBackup_Yes,DeviceProtection_No internet service,DeviceProtection_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes
0,7590-VHVEG,Female,0,Yes,No,0.013889,No,Yes,0.115423,0.001275,...,1,0,0,1,0,0,0,0,0,0
1,5575-GNVDE,Male,0,No,No,0.472222,Yes,No,0.385075,0.215867,...,0,0,0,0,0,1,0,0,0,0
2,3668-QPYBK,Male,0,No,No,0.027778,Yes,Yes,0.354229,0.010310,...,0,0,0,1,0,0,0,0,0,0
3,7795-CFOCW,Male,0,No,No,0.625000,No,No,0.239303,0.210241,...,1,0,0,0,0,1,0,0,0,0
4,9237-HQITU,Female,0,No,No,0.027778,Yes,Yes,0.521891,0.015330,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,0.333333,Yes,Yes,0.662189,0.227521,...,0,1,0,0,0,1,0,1,0,1
7039,2234-XADUH,Female,0,Yes,Yes,1.000000,Yes,Yes,0.845274,0.847461,...,0,1,0,1,0,1,0,1,0,1
7040,4801-JZAZL,Female,0,Yes,Yes,0.152778,No,Yes,0.112935,0.037809,...,1,0,0,0,0,0,0,0,0,0
7041,8361-LTMKD,Male,1,Yes,No,0.055556,Yes,Yes,0.558706,0.033210,...,0,1,0,0,0,0,0,0,0,0


In [298]:
colunas_binarias = [
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling",
    "Churn",
]
for coluna in colunas_binarias:
    print(f"\nContagem para coluna: {coluna}")
    print(dados_encoded[coluna].value_counts())


Contagem para coluna: Partner
Partner
No     3641
Yes    3402
Name: count, dtype: int64

Contagem para coluna: Dependents
Dependents
No     4933
Yes    2110
Name: count, dtype: int64

Contagem para coluna: PhoneService
PhoneService
Yes    6361
No      682
Name: count, dtype: int64

Contagem para coluna: PaperlessBilling
PaperlessBilling
Yes    4171
No     2872
Name: count, dtype: int64

Contagem para coluna: Churn
Churn
No     5174
Yes    1869
Name: count, dtype: int64


In [299]:
#transformando as colunas binárias para o formato de 0 e 1, onde Yes é 1 e No é 0.
for coluna in colunas_binarias:
    dados_encoded[coluna] = (
        dados_encoded[coluna]
        .map({"Yes": 1, "No": 0})
        .astype("int8")
    )

In [300]:
#Fazendo a contagem novamente para validar a transformação
for coluna in colunas_binarias:
    print(f"\nContagem para coluna: {coluna}")
    print(dados_encoded[coluna].value_counts())


Contagem para coluna: Partner
Partner
0    3641
1    3402
Name: count, dtype: int64

Contagem para coluna: Dependents
Dependents
0    4933
1    2110
Name: count, dtype: int64

Contagem para coluna: PhoneService
PhoneService
1    6361
0     682
Name: count, dtype: int64

Contagem para coluna: PaperlessBilling
PaperlessBilling
1    4171
0    2872
Name: count, dtype: int64

Contagem para coluna: Churn
Churn
0    5174
1    1869
Name: count, dtype: int64


In [301]:
#Normalizando a coluna gender, onde male é 1 e female é 0.
dados_encoded["gender"] = dados_encoded["gender"].map({
    "Male": 1,
    "Female": 0
}).astype("int8")

In [302]:
#Removendo a coluna customerID, pois ela não é relevante para a análise e pode ser considerada um identificador único para cada cliente.
dados_encoded = dados_encoded.drop(columns=["customerID"])

In [303]:
service_columns = [
    "StreamingTV_Yes",
    "StreamingMovies_Yes",
    "OnlineBackup_Yes",
    "DeviceProtection_Yes",
    "TechSupport_Yes",
    "OnlineSecurity_Yes",
]

dados_encoded["NumServices"] = dados_encoded[service_columns].sum(axis=1)

In [304]:
dados_encoded.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,MultipleLines_Yes,OnlineBackup_No internet service,OnlineBackup_Yes,DeviceProtection_No internet service,DeviceProtection_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,NumServices
0,0,0,1,0,0.013889,0,1,0.115423,0.001275,0,...,0,0,1,0,0,0,0,0,0,1
1,1,0,0,0,0.472222,1,0,0.385075,0.215867,0,...,0,0,0,0,1,0,0,0,0,2
2,1,0,0,0,0.027778,1,1,0.354229,0.010310,1,...,0,0,1,0,0,0,0,0,0,2
3,1,0,0,0,0.625000,0,0,0.239303,0.210241,0,...,0,0,0,0,1,0,0,0,0,3
4,0,0,0,0,0.027778,1,1,0.521891,0.015330,1,...,0,0,0,0,0,0,0,0,0,0


In [305]:
dados_encoded.to_csv(
    "../data/processed/telco_churn_encoded.csv",
    index=False
)